In [1]:
import geopandas as gpd
import numpy as np
import pandas as pd
import geojson_validator
from shapely.ops import unary_union
import pandas as pd
import numpy as np
from shapely.geometry import Point, MultiPolygon
import geopandas as gpd
from geopandas import GeoDataFrame
from fuzzywuzzy import process

In [17]:
block_gdf = gpd.read_file(r"D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\flood-data-ecosystem-Odisha\Maps\od_ids-drr_shapefiles\odisha_block_final_max-reduced.geojson")
block_ref = gpd.read_file(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\flood-data-ecosystem-Odisha\Maps\od_ids-drr_shapefiles\odisha_block_final.geojson')
risk_df = pd.read_csv(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\flood-data-ecosystem-Odisha\RiskScoreModel\data\risk_score.csv')

In [4]:
block_ref

,block_lgd,dtname,dtcode11,block_name,object_id,block_area,geometry
0,03276,Anugul,384,ANUGUL,21-384-03276,1116,"MULTIPOLYGON (((84.82174 20.56003, 84.82096 20..."
1,03277,Anugul,384,ATHMALLIK,21-384-03277,1027,"POLYGON ((84.35602 20.77060, 84.35411 20.77334..."
2,03278,Anugul,384,BANARPAL,21-384-03278,257,"MULTIPOLYGON (((85.08249 20.83019, 85.08400 20..."
3,03279,Anugul,384,CHHENDIPADA,21-384-03279,954,"MULTIPOLYGON (((84.79222 20.87905, 84.79139 20..."
4,03280,Anugul,384,KANIHA,21-384-03280,713,"POLYGON ((84.98528 21.01269, 84.98529 21.01286..."
...,...,...,...,...,...,...,...
309,03585,Sundargarh,374,NUAGAON,21-374-03585,434,"POLYGON ((84.84922 22.34176, 84.84737 22.34209..."
310,03586,Sundargarh,374,RAJGANGPUR,21-374-03586,583,"POLYGON ((84.51444 22.05402, 84.51130 22.05467..."
311,03587,Sundargarh,374,SUBDEGA,21-374-03587,381,"POLYGON ((84.07057 22.11483, 84.07027 22.11518..."
312,03588,Sundargarh,374,SUNDARGARH,21-374-03588,380,"POLYGON ((84.05317 21.93853, 84.05298 21.93870..."


In [18]:
block_merge = pd.merge(block_gdf, block_ref[['object_id','block']],on='block',how='left')
block_gdf = block_merge

KeyError: "['block'] not in index"

In [19]:
block_gdf

,block_lgd,dtname,dtcode11,block_name,object_id,geometry
0,03276,Anugul,384,ANUGUL,21-384-03276,"POLYGON ((84.79010 20.58127, 84.84852 20.52803..."
1,03277,Anugul,384,ATHMALLIK,21-384-03277,"POLYGON ((84.32141 20.85289, 84.35542 20.81894..."
2,03278,Anugul,384,BANARPAL,21-384-03278,"POLYGON ((85.03052 20.83493, 85.09557 20.85139..."
3,03279,Anugul,384,CHHENDIPADA,21-384-03279,"POLYGON ((84.78059 20.89691, 84.81686 20.85508..."
4,03280,Anugul,384,KANIHA,21-384-03280,"POLYGON ((84.96277 21.17580, 84.95632 21.08625..."
...,...,...,...,...,...,...
309,03585,Sundargarh,374,NUAGAON,21-374-03585,"POLYGON ((84.74924 22.41905, 84.82286 22.35122..."
310,03586,Sundargarh,374,RAJGANGPUR,21-374-03586,"POLYGON ((84.45176 22.18435, 84.47636 22.19000..."
311,03587,Sundargarh,374,SUBDEGA,21-374-03587,"POLYGON ((84.05082 22.13313, 84.09999 22.07704..."
312,03588,Sundargarh,374,SUNDARGARH,21-374-03588,"POLYGON ((84.02877 21.95288, 84.07861 21.93751..."


In [20]:
# Function to fix invalid geometries
def fix_geometry(geom):
    if geom is None:
        return None
    if not geom.is_valid:
        # Attempt to fix using buffer(0)
        geom = geom.buffer(0)
    return geom if geom.is_valid else None

# Check and fix geometries
def clean_geometries(gdf):
    # Check for missing or invalid geometries
    gdf['geometry_fixed'] = gdf['geometry'].apply(fix_geometry)

    # Drop rows with irreparable (None) geometries
    gdf = gdf.dropna(subset=['geometry_fixed'])

    return gdf

def geometry_to_text(geom):
    if geom is None:
        return None
    
    # Handle Polygon and LineString geometries
    if geom.geom_type == 'Polygon':
        coords = list(geom.exterior.coords)
        return str([[lon, lat] for lon, lat in coords])
    
    elif geom.geom_type == 'LineString':
        coords = list(geom.coords)
        return str([[lon, lat] for lon, lat in coords])
    
    # Handle MultiPolygon and MultiLineString geometries
    elif geom.geom_type in ['MultiPolygon', 'MultiLineString']:
        all_coords = []
        for part in geom.geoms:  # Loop through each sub-geometry
            if part.geom_type == 'Polygon':
                coords = list(part.exterior.coords)
            else:
                coords = list(part.coords)
            all_coords.append([[lon, lat] for lon, lat in coords])
        return str(all_coords)
    
    # Catch other geometry types if necessary
    else:
        return None

def get_best_match(block_name, block_names):
    match, score = process.extractOne(block_name, block_names)
    return match if score > 80 else None  # Adjust threshold as needed

In [21]:
import json

# Flatten nested geometries if required
fixed_geo_flattened = {
    key: [json.dumps(item) if isinstance(item, dict) else item for item in value]
    for key, value in block_gdf.items()
}
geo_fixed = pd.DataFrame.from_dict(fixed_geo_flattened)

In [22]:
# Simplify MultiPolygon by selecting the largest Polygon
def simplify_multipolygon(geometry):
    if isinstance(geometry, MultiPolygon):
        # Select the largest Polygon by area
        return max(geometry.geoms, key=lambda geom: geom.area)
    return geometry

# Apply the simplification
block_gdf['geometry'] = block_gdf['geometry'].apply(simplify_multipolygon)

print(type(gpd))
geo_fixed = gpd.GeoDataFrame(block_gdf, geometry='geometry')


<class 'module'>


In [23]:
merged_gdf = risk_df.merge(geo_fixed[['geometry','object_id']], left_on='object-id', right_on='object_id', how='left')
#merged_gdf = merged_gdf.drop(columns=['dtcode11','block_lgd','dtname','block_name'])
merged_gdf

,object-id,block-area,district,timeperiod,total-tender-awarded-value,ridf-tenders-awarded-value,preparedness-measures-tenders-awarded-value,immediate-measures-tenders-awarded-value,others-tenders-awarded-value,id,...,distance-from-river,exposure,government-response,flood-hazard,vulnerability,efficiency,topsis-score,risk-score,geometry,object_id
0,21-396-03557,825,Rayagada,2021_04,1872762.65,0.0,0.0,0.0,0.0,0,...,1953.231762,3,2,3.0,5,0.673123,0.737464,5,"POLYGON ((83.22666 19.17329, 83.19519 19.07634...",21-396-03557
1,21-375-03452,568,Kendujhar,2021_04,0.00,0.0,0.0,0.0,0.0,0,...,2583.572863,3,2,3.0,5,0.705012,0.737464,5,"POLYGON ((85.48243 21.74531, 85.49072 21.68958...",21-375-03452
2,21-398-03472,474,Koraput,2021_04,0.00,0.0,0.0,0.0,0.0,0,...,2484.077693,2,2,3.0,5,0.691333,0.640475,5,"POLYGON ((82.24013 18.95558, 82.27751 18.96326...",21-398-03472
3,21-397-03516,655,Nabarangpur,2021_04,0.00,0.0,0.0,0.0,0.0,0,...,2527.002772,2,2,3.0,5,0.686030,0.640475,5,"POLYGON ((82.16821 19.43672, 82.18927 19.41536...",21-397-03516
4,21-399-03480,1205,Malkangiri,2021_04,0.00,0.0,0.0,0.0,0.0,0,...,2721.758024,2,2,3.0,4,0.760226,0.594490,5,"POLYGON ((81.58276 17.83625, 81.61581 17.82198...",21-399-03480
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13811,21-377-03298,238,Baleshwar,2024_11,0.00,0.0,0.0,0.0,0.0,0,...,3332.705214,1,2,2.0,1,1.000000,0.208772,1,"POLYGON ((86.72423 21.31792, 86.76954 21.22556...",21-377-03298
13812,21-381-03338,119,Cuttack,2024_11,0.00,0.0,0.0,0.0,0.0,0,...,1872.991598,1,2,2.0,1,1.000000,0.208772,1,"POLYGON ((85.95061 20.27001, 86.00558 20.21748...",21-381-03338
13813,21-386-03462,114,Khordha,2024_11,0.00,0.0,0.0,0.0,0.0,0,...,4416.370611,1,2,2.0,1,1.000000,0.208772,1,"POLYGON ((85.14360 19.72996, 85.18250 19.67223...",21-386-03462
13814,21-389-03359,225,Gajapati,2024_11,0.00,0.0,0.0,0.0,0.0,0,...,3729.770241,1,2,2.0,1,1.000000,0.208772,1,"POLYGON ((83.98113 18.89098, 83.91758 18.84380...",21-389-03359


In [25]:
merged_gdf['polygons'] = merged_gdf['geometry'].apply(geometry_to_text)
merged_gdf

,object-id,block-area,district,timeperiod,total-tender-awarded-value,ridf-tenders-awarded-value,preparedness-measures-tenders-awarded-value,immediate-measures-tenders-awarded-value,others-tenders-awarded-value,id,...,exposure,government-response,flood-hazard,vulnerability,efficiency,topsis-score,risk-score,geometry,object_id,polygons
0,21-396-03557,825,Rayagada,2021_04,1872762.65,0.0,0.0,0.0,0.0,0,...,3,2,3.0,5,0.673123,0.737464,5,"POLYGON ((83.22666 19.17329, 83.19519 19.07634...",21-396-03557,"[[83.22665926315767, 19.173292819854517], [83...."
1,21-375-03452,568,Kendujhar,2021_04,0.00,0.0,0.0,0.0,0.0,0,...,3,2,3.0,5,0.705012,0.737464,5,"POLYGON ((85.48243 21.74531, 85.49072 21.68958...",21-375-03452,"[[85.48243108901275, 21.74531310193701], [85.4..."
2,21-398-03472,474,Koraput,2021_04,0.00,0.0,0.0,0.0,0.0,0,...,2,2,3.0,5,0.691333,0.640475,5,"POLYGON ((82.24013 18.95558, 82.27751 18.96326...",21-398-03472,"[[82.2401278874084, 18.955579791259787], [82.2..."
3,21-397-03516,655,Nabarangpur,2021_04,0.00,0.0,0.0,0.0,0.0,0,...,2,2,3.0,5,0.686030,0.640475,5,"POLYGON ((82.16821 19.43672, 82.18927 19.41536...",21-397-03516,"[[82.16820795458308, 19.436723947611878], [82...."
4,21-399-03480,1205,Malkangiri,2021_04,0.00,0.0,0.0,0.0,0.0,0,...,2,2,3.0,4,0.760226,0.594490,5,"POLYGON ((81.58276 17.83625, 81.61581 17.82198...",21-399-03480,"[[81.58275802541453, 17.836247025984328], [81...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13811,21-377-03298,238,Baleshwar,2024_11,0.00,0.0,0.0,0.0,0.0,0,...,1,2,2.0,1,1.000000,0.208772,1,"POLYGON ((86.72423 21.31792, 86.76954 21.22556...",21-377-03298,"[[86.72422805210445, 21.317917035134162], [86...."
13812,21-381-03338,119,Cuttack,2024_11,0.00,0.0,0.0,0.0,0.0,0,...,1,2,2.0,1,1.000000,0.208772,1,"POLYGON ((85.95061 20.27001, 86.00558 20.21748...",21-381-03338,"[[85.95060569912471, 20.270008590798955], [86...."
13813,21-386-03462,114,Khordha,2024_11,0.00,0.0,0.0,0.0,0.0,0,...,1,2,2.0,1,1.000000,0.208772,1,"POLYGON ((85.14360 19.72996, 85.18250 19.67223...",21-386-03462,"[[85.14359737057575, 19.729955521145445], [85...."
13814,21-389-03359,225,Gajapati,2024_11,0.00,0.0,0.0,0.0,0.0,0,...,1,2,2.0,1,1.000000,0.208772,1,"POLYGON ((83.98113 18.89098, 83.91758 18.84380...",21-389-03359,"[[83.98112873028782, 18.89097889522747], [83.9..."


In [26]:
merged_gdf.to_csv(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\risk-score\Odisha\Od_risk.csv')

### Datetime ISO edits

In [12]:
from datetime import date, timedelta, datetime


#snapshot_ld_dist['datetime'] = pd.to_datetime(snapshot_ld_dist['timeperiod'], format='%Y_%m')
merged['timeperiod_iso'] = merged['timeperiod'].str.replace('_', '-') #+ '-01'

# Step 2: Convert the modified column to datetime format
#snapshot_ld_dist['timeperiod_iso'] = pd.to_datetime(snapshot_ld_dist['timeperiod_iso'], format='%Y-%m-%d')
merged['timeperiod_iso'] = pd.to_datetime(merged['timeperiod_iso'], format='%Y-%m')


merged['timeperiod_iso'] = merged['timeperiod_iso'].dt.strftime('%Y-%m')

In [14]:
merged.to_csv(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\risk-score\risk_score_polygons.csv')